# Chapter 2: Vision Backbone
Compare scratch CNN vs frozen CLIP vs frozen SigLIP on MiniPushT 224x224.

In [ ]:
!pip install torch torchvision numpy gymnasium matplotlib transformers scikit-learn

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/02_vision_backbone

## Explore the 224x224 Environment

In [ ]:
from mini_pusht import MiniPushT
import matplotlib.pyplot as plt

env = MiniPushT()
obs, info = env.reset(seed=42)
plt.figure(figsize=(4, 4))
plt.imshow(obs)
plt.title(f"MiniPushT 224x224\nagent={info['agent_pos']}, block={info['block_pos']}, goal={info['goal_pos']}")
plt.axis("off")
plt.show()
print(f"Observation shape: {obs.shape}, dtype: {obs.dtype}")

## Collect Expert Demos

In [ ]:
from compare_encoders import ScriptedExpert, collect_demos

expert = ScriptedExpert()
demos = collect_demos(env, expert, n_episodes=1000)
print(f"Collected {len(demos)} transitions from 1000 episodes")
print(f"Image shape: {demos[0]['image'].shape}")

## Train All Three Encoders

In [ ]:
import time
import torch
from compare_encoders import (
    ScratchCNN, PretrainedVisionEncoder, VLA, PushTDataset,
    train, evaluate, compute_embeddings_for_tsne
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

dataset = PushTDataset(demos)

encoder_configs = [
    ("Scratch CNN", ScratchCNN(), 128),
    ("CLIP ViT-B/16", PretrainedVisionEncoder("openai/clip-vit-base-patch16"), 768),
    ("SigLIP ViT-B/16", PretrainedVisionEncoder("google/siglip-base-patch16-224"), 768),
]

results = {}
losses_dict = {}
embeddings_dict = {}
shared_labels = None

for name, encoder, dim in encoder_configs:
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    model = VLA(vision_encoder=encoder, vision_dim=dim)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {n_train:,}")

    start = time.time()
    losses = train(model, dataset, epochs=20, device=device)
    elapsed = time.time() - start

    rate = evaluate(model, env, n_episodes=50, device=device)
    results[name] = {"params": n_train, "time": elapsed, "loss": losses[-1], "rate": rate}
    losses_dict[name] = losses

    embeds, labels = compute_embeddings_for_tsne(model, dataset, n_samples=500, device=device)
    embeddings_dict[name] = embeds
    if shared_labels is None:
        shared_labels = labels

print(f"\n{'='*50}")
print("RESULTS")
print(f"{'='*50}")
for name, r in results.items():
    print(f"{name:<20} {r['params']:>10,} params  {r['time']:>6.1f}s  loss={r['loss']:.4f}  success={r['rate']*100:.1f}%")

## Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, losses in losses_dict.items():
    ax.plot(range(1, len(losses)+1), losses, marker="o", markersize=3, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss by vision encoder")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## t-SNE Embedding Visualization

In [ ]:
from compare_encoders import plot_tsne_comparison
plot_tsne_comparison(embeddings_dict, shared_labels, save_path="/tmp/ch02_tsne.png")
from IPython.display import Image
Image("/tmp/ch02_tsne.png")

## What We Learned

Pretrained vision encoders (CLIP, SigLIP) produce features that already capture spatial relationships useful for action prediction -- without seeing a single MiniPushT frame during pretraining.

**Next:** Chapter 3 swaps the lookup-table language encoder for SmolLM2.